# TerraMatch Chile — Análisis exploratorio

Notebook reproducible del modelo (secciones 6, 8 y 10.5 del documento v2.0). Las reglas se leen de `config/terramatch_config.json`.

**Regla de integridad:** este notebook solo muestra resultados calculados con los datos descargados. Si faltan datos, lo indica y no calcula nada. Nunca se ajustan los resultados para que coincidan con lo esperado.

*AI-Assisted: Claude (borrador del notebook); revisado y ejecutado por el equipo.*

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from config import CFG, VARS
import analysis as an, comparison as cp, validation as vl
from run_pipeline import run
pd.DataFrame([{"variable": v, **{k: CFG["variables"][v][k] for k in ("label", "unit", "weight", "transform")}} for v in VARS])

## 1. Parámetros del modelo

In [ ]:
pd.Series({
    "Resolución común (m)": round(CFG["common_resolution_m"], 1),
    "Ventana local (km)": CFG["window_km"],
    "Grilla": f'{CFG["grid"]["crs"]}, {CFG["grid"]["resolution_deg"]}°',
    "Cobertura mínima": "2 de 3 variables",
    "Iteraciones Monte Carlo": CFG["monte_carlo"]["iterations"],
    "Filtro Marte": CFG["filters"]["mars"]["label"],
    "Filtro Luna": CFG["filters"]["moon"]["label"],
}, name="valor").to_frame()

## 2. Estado de los datos de entrada

In [ ]:
paths = {"grid": ROOT/"data/processed/grid.csv", "master": ROOT/"data/processed/master.csv", "targets": ROOT/"data/targets.json"}
grid = pd.read_csv(paths["grid"]) if paths["grid"].exists() else None
master = pd.read_csv(paths["master"]) if paths["master"].exists() else None
targets = an.load_targets(paths["targets"]) if paths["targets"].exists() else {}
status = [{"insumo": "Grilla nacional", "estado": f"{len(grid):,} celdas" if grid is not None else "falta"},
          {"insumo": "Variables de Chile (master.csv)", "estado": "lista" if master is not None else "falta: ASTER e IMERG (Pancho)"}]
for tid, t in targets.items():
    pend = an.pending_variables(t)
    status.append({"insumo": f"Perfil {t['name']}", "estado": "listo" if not pend else f"pendiente de verificación: {', '.join(pend)}"})
ready = grid is not None and master is not None and any(not an.pending_variables(t) for t in targets.values())
pd.DataFrame(status)

In [ ]:
if grid is not None:
    fig, ax = plt.subplots(figsize=(3, 8))
    ax.scatter(grid.lon_c, grid.lat_c, s=0.2, c="0.4")
    ax.set(title=f"Grilla nacional ({len(grid):,} celdas)", xlabel="Longitud", ylabel="Latitud"); ax.set_aspect("equal")
    plt.show()

## 3. Distribución de las variables de relieve en Chile

In [ ]:
if master is not None:
    fig, axes = plt.subplots(1, 3, figsize=(12, 3))
    for ax, v in zip(axes, VARS):
        master[v].dropna().plot.hist(bins=60, ax=ax, color="0.4")
        ax.set(title=CFG["variables"][v]["label"], xlabel=CFG["variables"][v]["unit"])
    plt.tight_layout(); plt.show()
    display(master[list(VARS)].describe().T)
else:
    print("Sin master.csv todavía: falta correr src/features/chile_features.py con ASTER e IMERG.")

## 4. Similitud, estabilidad y validación

In [ ]:
if ready:
    report = run(grid, master, targets, ROOT/"results", sites_dir=ROOT/"data/chile/sites")
    print(json.dumps(report, indent=2, ensure_ascii=False))
else:
    report = None
    print("No se calcula: faltan datos o perfiles verificados (ver sección 2). No se generan resultados de ejemplo.")

In [ ]:
sites_path = ROOT/"results/sites_summary.csv"
if report is not None and sites_path.exists():
    sites = pd.read_csv(sites_path)
    display(sites[["site", "role", "profile", "S", "percentile", "p_top", "n_vars", "confidence", "method"]])
    fig, ax = plt.subplots(figsize=(8, 3))
    ok = sites.dropna(subset=["pct_p05", "pct_p95"])
    ax.errorbar(ok.site, ok.percentile, yerr=[ok.percentile - ok.pct_p05, ok.pct_p95 - ok.percentile], fmt="o", capsize=4)
    ax.axhline(75, ls="--", c="g", lw=1); ax.axhline(25, ls="--", c="r", lw=1)
    ax.set(ylabel="Percentil (100 = más similar)", title="Percentil de cada sitio y rango p05–p95 del Monte Carlo")
    plt.show()

In [ ]:
for body in ("mars", "moon"):
    p = ROOT/f"results/{body}_profile.csv"
    if report is not None and report.get(body, {}).get("status") == "computed" and p.exists():
        prof = pd.read_csv(p).merge(grid, on="cell_id")
        sc = prof[prof.status == "scored"]
        fig, ax = plt.subplots(figsize=(3, 8))
        ax.scatter(prof.lon_c, prof.lat_c, s=0.2, c="0.85")
        im = ax.scatter(sc.lon_c, sc.lat_c, s=0.3, c=sc.S, cmap="viridis", vmin=0, vmax=100)
        plt.colorbar(im, ax=ax, label="S (0-100)"); ax.set_aspect("equal")
        ax.set(title=f"Similitud con el perfil {body} (gris: fuera del filtro)"); plt.show()

## 5. Exploración complementaria (opcional): PCA y K-means

Agrupación estadística, no prueba de analogía (doc, Fase 4).

In [ ]:
try:
    from sklearn.decomposition import PCA
    from sklearn.cluster import KMeans
    from sklearn.metrics import silhouette_score
    have_sk = True
except ImportError:
    have_sk = False; print("scikit-learn no instalado: py -m pip install scikit-learn")
if have_sk and master is not None:
    X = master[list(VARS)].dropna()
    Z = (X - X.median()) / (X.quantile(.75) - X.quantile(.25))
    pca = PCA().fit(Z); print("Varianza explicada:", np.round(pca.explained_variance_ratio_, 3))
    sample = Z.sample(min(5000, len(Z)), random_state=0)
    sil = {k: silhouette_score(sample, KMeans(k, n_init=10, random_state=0).fit_predict(sample)) for k in range(3, 9)}
    print("Silueta por k:", {k: round(v, 3) for k, v in sil.items()}, "-> k =", max(sil, key=sil.get))